# Week 6: Check your results and finish the case study

This is the final portfolio week. Your goal is to make the analysis **trustworthy and understandable to someone who did not build it**. You will check the cleaned data, calculate a small set of clearly defined metrics, and write a short case study in [`reports/final_case_study.md`](../reports/final_case_study.md).

**You will write code.** Each exercise introduces a Python idea, shows a small example, then asks you to use it on the Online Retail data. Replace `None` with your own expression. Run cells in order with **Shift+Enter**; save with **⌘S**. If you restart the kernel, run from Section 1 again. The notebook prints a friendly reminder when an exercise is still blank.

**Definitions to keep consistent:** One row is a product line. One order is a distinct `InvoiceNo`. Revenue is `Quantity × UnitPrice` on the positive-sales lines you kept in Week 2. This is not net revenue or profit. A repeat customer has at least two distinct positive-sales invoices and a recorded customer ID.

**Deliverables:** this completed notebook, `reports/week6_key_metrics.csv`, your written case study, and the Week 5 dashboard. You can ask for a review before uploading the final work to GitHub.

## 1. Load the Week 2 cleaned data

The setup code finds the project folder so the notebook works whether VS Code starts in the project root or in `notebooks/`. `dtype` keeps IDs as labels; `parse_dates` makes the invoice date usable as a date.

Run this cell. If the CSV is missing, run Week 2 Section 7 first.

In [1]:
from pathlib import Path
import pandas as pd

project_root = next(
    (folder for folder in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (folder / "data" / "processed" / "valid_sales.csv").is_file()),
    None,
)
if project_root is None:
    raise FileNotFoundError("Run Week 2 Section 7 to create data/processed/valid_sales.csv first.")

sales = pd.read_csv(
    project_root / "data" / "processed" / "valid_sales.csv",
    dtype={"InvoiceNo": "string", "StockCode": "string", "CustomerID": "string"},
    parse_dates=["InvoiceDate"],
)
reports_dir = project_root / "reports"
reports_dir.mkdir(exist_ok=True)

print(f"Loaded {len(sales):,} positive-sales lines")
print("Date range:", sales["InvoiceDate"].min(), "to", sales["InvoiceDate"].max())
display(sales[["InvoiceNo", "Quantity", "UnitPrice", "Revenue", "CustomerID"]].head(3))

Loaded 530,104 positive-sales lines
Date range: 2010-12-01 08:26:00 to 2011-12-09 12:50:00


,InvoiceNo,Quantity,UnitPrice,Revenue,CustomerID
0,536365,6,2.55,15.30,17850
1,536365,6,3.39,20.34,17850
2,536365,8,2.75,22.00,17850


**Write one sentence:** Why is `len(sales)` a line count rather than an order count? **___**

## 2. Audit the cleaned data

Before you publish numbers, check the rules you applied. A comparison such as `sales["Quantity"] > 0` produces one `True` or `False` per row. `.all()` returns `True` only when **every** row passes.

Worked example:

```python
quantity_ok = (sales["Quantity"] > 0).all()
```

Now check `UnitPrice` the same way. Then verify that saved `Revenue` still equals `Quantity * UnitPrice`. Floating point calculations can have tiny rounding differences, so use the **absolute gap** and allow less than one penny:

```python
revenue_gap = (sales["Revenue"] - sales["Quantity"] * sales["UnitPrice"]).abs()
revenue_ok = (revenue_gap < 0.01).all()
```

Type those expressions yourself below. If a check is `False`, investigate before using the dataset in your report.

In [2]:
quantity_ok = (sales["Quantity"] > 0).all()  # Worked example

# Your turn: replace None in all three lines.
price_ok = (sales["UnitPrice"]>0).all()
revenue_gap = (sales["Revenue"] - sales["Quantity"] * sales["UnitPrice"]).abs()
revenue_ok = (revenue_gap < 0.01).all()

if price_ok is None or revenue_gap is None or revenue_ok is None:
    print("Finish the price and revenue checks above, then run this cell again.")
else:
    print("All quantities positive:", quantity_ok)
    print("All unit prices positive:", price_ok)
    print("All line revenues match quantity × price within £0.01:", revenue_ok)
    print("Largest absolute revenue gap:", revenue_gap.max())

All quantities positive: True
All unit prices positive: True
All line revenues match quantity × price within £0.01: True
Largest absolute revenue gap: 0.0


**Explain the check:** Why do we use `.all()` here instead of `.sum()`? What does a `False` result mean? 

We want to use all to ensure that every row fits the condition, in which case it does. False here means at least one of the rows did not meet our condition.

## 3. Define your headline metrics

Here are the questions the final report needs to answer:

- **Positive-sales revenue:** add `Revenue` over all retained lines.
- **Orders:** count distinct `InvoiceNo` values with `.nunique()`.
- **Average order value:** divide total revenue by distinct orders, **not** by product lines.
- **Identified customers:** count distinct nonmissing `CustomerID` values with `.nunique()`.

Worked example: `total_revenue = sales["Revenue"].sum()`.

**Your turn:** Fill in the other three variables. Calculate `order_count` before `average_order_value` because the latter needs it. A variable stores a result so you can reuse it later.

In [3]:
total_revenue = sales["Revenue"].sum()  # Worked example

# Your turn: replace None on each line.
order_count = sales["InvoiceNo"].nunique()
average_order_value = total_revenue/order_count
identified_customer_count = sales["CustomerID"].nunique()

if any(value is None for value in [order_count, average_order_value, identified_customer_count]):
    print("Calculate orders, average order value, and identified customers above.")
else:
    print(f"Positive-sales revenue: £{total_revenue:,.2f}")
    print(f"Distinct orders: {order_count:,}")
    print(f"Average order value: £{average_order_value:,.2f}")
    print(f"Identified customers: {identified_customer_count:,}")

Positive-sales revenue: £10,666,684.54
Distinct orders: 19,960
Average order value: £534.40
Identified customers: 4,338


**Sanity check:** Your order count should be much smaller than the line count. Why? Is average order value the same as the average **product-line** revenue? **___**

## 4. Compare complete months fairly

Group by `YearMonth` and sum `Revenue`, as you did in Weeks 4 and 5. `.sort_index()` puts month labels in time order.

December 2011 contains only **December 1–9**. To find the highest **full** month, leave `2011-12` out of this comparison only. `.drop("2011-12")` removes that month from a Series without changing `sales`.

Two new methods:

```python
example = pd.Series({"Jan": 10, "Feb": 20})
example.idxmax()  # "Feb": the label of the largest value
example.max()     # 20: the largest value
```

**Your turn:** Create `monthly_revenue`, then `full_months`, then its highest month label and amount. Use the four expressions described above. Do not compare the partial December total as though it covered a full month.

In [10]:
# Your turn: replace None in all four lines.
monthly_revenue = sales.groupby("YearMonth")["Revenue"].sum().sort_index()
full_months = sales.groupby("YearMonth")["Revenue"].sum().sort_index().drop("2011-12")
highest_full_month = full_months.idxmax()
highest_full_month_revenue = full_months.max()

if any(value is None for value in
       [monthly_revenue, full_months, highest_full_month, highest_full_month_revenue]):
    print("Complete the monthly series and highest-full-month lines above.")
else:
    display(monthly_revenue.tail(4))
    print(f"Highest full month: {highest_full_month}, £{highest_full_month_revenue:,.2f}")
    print(f"All monthly totals: £{monthly_revenue.sum():,.2f}; compare with total revenue.")

YearMonth
2011-09    1058590.172
2011-10    1154979.300
2011-11    1509496.330
2011-12     638792.680
Name: Revenue, dtype: float64

Highest full month: 2011-11, £1,509,496.33
All monthly totals: £10,666,684.54; compare with total revenue.


**Interpretation:** Did you find the same highest full month as in the Week 5 memo? Explain why the last point of your dashboard should not be read as a confirmed sales decline.

Because the last point represents an incomplete December month, so we cannot fully confirm its efficacy.

## 5. Check the repeat-customer result

A customer can have several product lines on one invoice. To count their **orders**, first remove rows with no customer ID, group by `CustomerID`, then count **distinct** invoice numbers.

Try this tiny worked example. Customer A has two lines on invoice 1 and one line on invoice 2, so A has **two orders**, not three. The missing ID is not a trackable customer.

In [13]:
practice = pd.DataFrame({
    "CustomerID": ["A", "A", "A", "B", None],
    "InvoiceNo": ["1", "1", "2", "3", "4"],
})
display(practice)
display(practice.dropna(subset=["CustomerID"]).groupby("CustomerID")["InvoiceNo"].nunique())

,CustomerID,InvoiceNo
0,A,1
1,A,1
2,A,2
3,B,3
4,NaN,4


CustomerID
A    2
B    1
Name: InvoiceNo, dtype: int64

Read the chain from left to right:

1. `.dropna(subset=["CustomerID"])` removes untrackable IDs **for this customer calculation**. Those sales lines still count in total revenue.
2. `.groupby("CustomerID")` creates one group per known customer.
3. `["InvoiceNo"].nunique()` counts each distinct order once per customer.

**Your turn:** Apply this chain to `sales` and call the result `customer_orders`. Then count customers with at least two orders and divide by the number of identified customers. `.sum()` counts the `True` values in a Boolean comparison. For example, `(customer_orders >= 2).sum()` counts repeat customers.

In [20]:
# Your turn: replace None in all three lines.
customer_orders = sales.dropna(subset="CustomerID").groupby("CustomerID")["InvoiceNo"].nunique()
repeat_customer_count = (customer_orders >= 2).sum()
repeat_customer_rate = repeat_customer_count/(customer_orders).sum()  # a fraction between 0 and 1

display(customer_orders)
display(repeat_customer_count)

if any(value is None for value in
       [customer_orders, repeat_customer_count, repeat_customer_rate]):
    print("Complete the customer order count and repeat rate above.")
else:
    print(f"Known customers: {len(customer_orders):,}")
    print(f"Repeat customers: {repeat_customer_count:,}")
    print(f"Repeat rate: {repeat_customer_rate:.1%}")

CustomerID
12346     1
12347     7
12348     4
12349     1
12350     1
         ..
18280     1
18281     1
18282     2
18283    16
18287     3
Name: InvoiceNo, Length: 4338, dtype: int64

np.int64(2845)

Known customers: 4,338
Repeat customers: 2,845
Repeat rate: 15.4%


**Explain the denominator:** Does the repeat rate describe every person who bought from the retailer, or only customers with recorded IDs during this observation period? Why can’t it measure lifetime retention?

Only customers whose ID wasn't missing.

## 6. Export a small metrics table

A portfolio reader should be able to find your key numbers without searching through every notebook. A pandas `DataFrame` can hold a small table. This worked example shows the syntax:

```python
example_table = pd.DataFrame([
    {"Metric": "Invoice lines", "Value": len(sales)},
    {"Metric": "Orders", "Value": order_count},
])
```

Each `{...}` is one row. The keys become column names. Use this pattern to make `final_metrics` with **at least** revenue, orders, average order value, and repeat rate. A `Definition` column is helpful: write what each number counts. `to_csv(..., index=False)` saves the table without an extra row-number column.

The save code runs only after you replace `None`. If a needed metric is still `None`, finish the earlier section first.

In [22]:
# Your turn: replace None with pd.DataFrame([...]) using your metric variables.
final_metrics = pd.DataFrame([
    {"Revenue": 100, "Orders": len(sales), "Average Order Value": 10, "Repeat Rate": 0.5},
    {"Revenue": 40, "Orders": len(sales), "Average Order Value": 12, "Repeat Rate": 0.32},
])

if final_metrics is None:
    print("Build a small DataFrame of your final metrics, then run this cell again.")
elif any(value is None for value in [order_count, average_order_value, repeat_customer_rate]):
    print("Complete Sections 3 and 5 before exporting the metric table.")
else:
    display(final_metrics)
    output_path = reports_dir / "week6_key_metrics.csv"
    final_metrics.to_csv(output_path, index=False)
    print("Saved:", output_path)

,Revenue,Orders,Average Order Value,Repeat Rate
0,100,530104,10,0.50
1,40,530104,12,0.32


Saved: /Users/abepark/Documents/Codex/2026-10-28/referenced-chatgpt-conversation-this-is-an/ecommerce-sales-analysis/reports/week6_key_metrics.csv


## 7. Write the final case study

Open [`reports/final_case_study.md`](../reports/final_case_study.md). It is a Markdown template: `#` makes a heading, `##` a section heading, and `![caption](path)` displays your dashboard image on GitHub. Replace the prompts with **your own words and numbers** from the notebook. Keep it short enough that a hiring manager can scan it.

Use this order:

1. **Question and data:** What business question did you answer, and where did the data come from?
2. **Method:** What did one row represent? Which sales lines were excluded? How did you define an order and a repeat customer?
3. **Findings:** Give three numbered findings with figures and plain-language meaning.
4. **Recommendation:** Suggest one specific next analysis or small test supported by a finding. Do not claim the data proves a cause.
5. **Limitations:** Mention the partial December, missing customer IDs, and the difference between positive-sales revenue and net revenue/profit.

**A useful writing test:** Can someone read each finding and answer “What number supports this?” and “Why should I care?”

## 8. Check the portfolio before you publish

- [ ] Every required code exercise runs from a fresh kernel in order. In VS Code: **Restart Kernel**, then **Run All**.
- [ ] The checks in Section 2 are all `True`, or you investigated any failure.
- [ ] Your Week 6 numbers agree with the Week 4 and Week 5 definitions.
- [ ] `reports/week6_key_metrics.csv` exists and has clear metric definitions.
- [ ] `reports/final_case_study.md` has no `___` prompts and shows the Week 5 dashboard.
- [ ] Your report cites the UCI Online Retail dataset and explains its limits.
- [ ] The raw workbook, processed CSV, local `.venv`, and database are **not** added to GitHub.

When you finish, save both files and ask for a review. We can then publish the final Week 6 work to your GitHub repository.